In [ ]:
!git pull

In [ ]:
import os, pathlib

# 1. Clone the repo first
%cd /kaggle/working
!git clone https://github.com/jo3591/assigment2dsai413 cxr-intel
%cd cxr-intel
!pip install -q -r requirements-colab.txt
!pip install -q -e .

# 2. Load secrets
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
os.environ['HF_TOKEN'] = secrets.get_secret('HF_TOKEN')
os.environ['NVIDIA_API_KEY'] = secrets.get_secret('NVIDIA_API_KEY')
os.environ['LLM_PROVIDER'] = 'nvidia'
os.environ['QA_SYNTH_MODEL'] = 'meta/llama-3.1-8b-instruct'
os.environ['QA_JUDGE_MODEL'] = 'meta/llama-3.3-70b-instruct'

# 3. Symlink dataset (now data/ exists because pip install created it via package install)
dst = pathlib.Path('/kaggle/working/cxr-intel/data/raw')
dst.parent.mkdir(parents=True, exist_ok=True)
if dst.exists() or dst.is_symlink():
    dst.unlink()
dst.symlink_to('/kaggle/input/datasets/simhadrisadaram/mimic-cxr-dataset')

# 4. Verify
import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")
print(f"Dataset contents: {os.listdir(dst)[:3]}")


In [ ]:
!pip uninstall -y bitsandbytes
!python -u scripts/build_indices.py --backend colpali_zs


In [ ]:
import os, pathlib
for p in ['data/processed/reports.parquet',
          'data/indices/biomedclip/index.faiss',
          'data/indices/colpali_zs/doc_embeddings.npy']:
    f = pathlib.Path(p)
    print(f"  {'✓' if f.exists() else '✗'} {p} ({f.stat().st_size/1e6:.1f} MB)" if f.exists() else f"  ✗ {p} MISSING")


In [ ]:
!python -u scripts/build_qa_dataset.py --config configs/data.yaml


In [ ]:
%cd /kaggle/working/cxr-intel
!pip install -q -e .


In [ ]:
import sys, os
sys.path.insert(0, '/kaggle/working/cxr-intel/src')
os.chdir('/kaggle/working/cxr-intel')

from cxr_intel.utils.io import load_jsonl
from collections import Counter

for name in ['qa_v1.jsonl', 'qa_test.jsonl']:
    path = f'data/qa/{name}'
    if os.path.exists(path):
        qa = list(load_jsonl(path))
        types = Counter(p['question_type'] for p in qa)
        labels = Counter(p['anchor_label'] for p in qa)
        print(f"{name}: {len(qa)} pairs | types={dict(types)}")
        print(f"  top labels: {dict(labels.most_common(5))}")
    else:
        print(f"{name}: MISSING")


In [ ]:
!pip install -q "bitsandbytes>=0.45,<0.50"


In [ ]:
!pip install -q --force-reinstall --no-deps "transformers>=4.50,<4.53"
!python -c "import transformers; print('Now installed:', transformers.__version__)"


In [ ]:
import os, sys, pathlib
sys.path.insert(0, '/kaggle/working/cxr-intel/src')
os.chdir('/kaggle/working/cxr-intel')

# Re-link the dataset
dst = pathlib.Path('/kaggle/working/cxr-intel/data/raw')
if not dst.exists():
    dst.symlink_to('/kaggle/input/datasets/simhadrisadaram/mimic-cxr-dataset')

# Re-set secrets/env
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
os.environ['HF_TOKEN'] = secrets.get_secret('HF_TOKEN')
os.environ['NVIDIA_API_KEY'] = secrets.get_secret('NVIDIA_API_KEY')
os.environ['LLM_PROVIDER'] = 'nvidia'
os.environ['QA_SYNTH_MODEL'] = 'meta/llama-3.1-8b-instruct'
os.environ['QA_JUDGE_MODEL'] = 'meta/llama-3.3-70b-instruct'

# Verify versions
from importlib.metadata import version
print(f"transformers: {version('transformers')}")   # should be 4.52.x
print(f"colpali-engine: {version('colpali-engine')}")

# Verify data still on disk
for p in ['data/processed/reports.parquet', 'data/qa/qa_v1.jsonl', 'data/qa/qa_test.jsonl',
          'data/indices/biomedclip/index.faiss', 'data/indices/colpali_zs/doc_embeddings.npy']:
    print(f"{'✓' if pathlib.Path(p).exists() else '✗'} {p}")

# Load MedGemma
import torch
print(f"\nGPU: {torch.cuda.get_device_name(0)}")
from cxr_intel.models.medgemma_runner import MedGemmaRunner
m = MedGemmaRunner(quantization='int4')
m.load()
print(f"✓ MedGemma loaded | VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB")


In [ ]:
import torch, safetensors.torch, shutil
from pathlib import Path
from huggingface_hub import snapshot_download

# 1. Download the original ColPali adapter
src_dir = Path(snapshot_download("vidore/colpali-v1.3", allow_patterns=["adapter_*"]))
print(f"Downloaded original adapter to: {src_dir}")

# 2. Patch the adapter weight keys (remap layer path for transformers 5.x)
state_dict = safetensors.torch.load_file(str(src_dir / "adapter_model.safetensors"))
patched = {
    k.replace("model.language_model.model.", "model.model.language_model."): v
    for k, v in state_dict.items()
}
print(f"Remapped {len(patched)} adapter keys")

# 3. Save patched adapter to local path
out_dir = Path("/kaggle/working/colpali-v1.3-patched")
out_dir.mkdir(exist_ok=True, parents=True)
safetensors.torch.save_file(patched, str(out_dir / "adapter_model.safetensors"))
shutil.copy(src_dir / "adapter_config.json", out_dir / "adapter_config.json")
print(f"✓ Patched adapter saved to: {out_dir}")

# 4. Test loading ColPali with the patched adapter (should see no UNEXPECTED/MISSING)
from colpali_engine.models import ColPali
model = ColPali.from_pretrained(
    str(out_dir),
    torch_dtype=torch.float16,
    device_map={"": "cuda:0"},
).eval()
print(f"✓ ColPali loaded — VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB")


In [ ]:
import os, sys, pathlib
sys.path.insert(0, '/kaggle/working/cxr-intel/src')
os.chdir('/kaggle/working/cxr-intel')

dst = pathlib.Path('/kaggle/working/cxr-intel/data/raw')
if not dst.exists():
    dst.symlink_to('/kaggle/input/datasets/simhadrisadaram/mimic-cxr-dataset')

from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
os.environ['HF_TOKEN'] = secrets.get_secret('HF_TOKEN')
os.environ['NVIDIA_API_KEY'] = secrets.get_secret('NVIDIA_API_KEY')
os.environ['LLM_PROVIDER'] = 'nvidia'

print(f"✓ patched ColPali dir exists: {pathlib.Path('/kaggle/working/colpali-v1.3-patched').exists()}")
print(f"✓ adapter file: {pathlib.Path('/kaggle/working/colpali-v1.3-patched/adapter_model.safetensors').exists()}")


In [ ]:
import shutil, safetensors.torch
from pathlib import Path
from huggingface_hub import snapshot_download

# 1. Download ALL files this time (not just adapter_*)
full_src = Path(snapshot_download("vidore/colpali-v1.3"))
print(f"Full snapshot at: {full_src}")
print(f"Contains: {sorted([p.name for p in full_src.iterdir()])}")

# 2. Make patched dir as a full copy of the snapshot
patched = Path("/kaggle/working/colpali-v1.3-patched")
if patched.exists():
    shutil.rmtree(patched)
# Don't use symlinks — we need writable files
patched.mkdir(parents=True)
for src in full_src.iterdir():
    if src.is_file():
        shutil.copy2(src, patched / src.name)
    else:
        shutil.copytree(src, patched / src.name)

# 3. Re-patch the adapter weights (overwrite the copied one)
adapter_path = patched / "adapter_model.safetensors"
state_dict = safetensors.torch.load_file(str(adapter_path))
remapped = {
    k.replace("model.language_model.model.", "model.model.language_model."): v
    for k, v in state_dict.items()
}
safetensors.torch.save_file(remapped, str(adapter_path))
print(f"Patched {len(remapped)} adapter keys in {patched}")

# 4. Verify all expected files are present
needed = ['adapter_model.safetensors', 'adapter_config.json',
          'preprocessor_config.json', 'tokenizer.json', 'tokenizer_config.json']
for f in needed:
    p = patched / f
    print(f"  {'✓' if p.exists() else '✗'} {f}")


In [ ]:
import os, sys, pathlib
sys.path.insert(0, '/kaggle/working/cxr-intel/src')
os.chdir('/kaggle/working/cxr-intel')

dst = pathlib.Path('/kaggle/working/cxr-intel/data/raw')
if not dst.exists():
    dst.symlink_to('/kaggle/input/datasets/simhadrisadaram/mimic-cxr-dataset')

from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
os.environ['HF_TOKEN'] = secrets.get_secret('HF_TOKEN')
os.environ['NVIDIA_API_KEY'] = secrets.get_secret('NVIDIA_API_KEY')
os.environ['LLM_PROVIDER'] = 'nvidia'

# Verify patched dir exists from earlier cell
print(f"✓ patched dir: {pathlib.Path('/kaggle/working/colpali-v1.3-patched/adapter_model.safetensors').exists()}")


In [ ]:
import os
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'   # reduce fragmentation

import pandas as pd, torch
from cxr_intel.retrieval.colpali_index import ColPaliRetriever

print(f"Starting VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB")

df = pd.read_parquet('data/processed/reports.parquet')
items = [
    {"study_id": str(r["study_id"]),
     "image_path": str(r["image_path"]),
     "report_text": (str(r.get("findings","")) + " " + str(r.get("impression",""))).strip()}
    for _, r in df.iterrows()
]

r = ColPaliRetriever(
    checkpoint="/kaggle/working/colpali-v1.3-patched",
    torch_dtype="float16",
    device_map="auto",
    batch_size=2,
)
r.index(items, "data/indices/colpali_zs")
print(f"✓ Done. VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB")


In [ ]:
from pathlib import Path
script = Path('scripts/run_eval.py')
content = script.read_text()
patched_content = content.replace(
    'r = ColPaliRetriever(name="colpali_zs")',
    'r = ColPaliRetriever(name="colpali_zs", checkpoint="/kaggle/working/colpali-v1.3-patched")'
)
if patched_content != content:
    script.write_text(patched_content)
    print("✓ Patched run_eval.py to use the local patched ColPali checkpoint")
else:
    print("⚠ No replacement made — check the source line manually")


In [ ]:
import gc, torch
for name in [n for n, o in list(globals().items()) if isinstance(o, torch.nn.Module)]:
    del globals()[name]
for name in ['r', 'model', 'mg', 'cp', 'm']:
    if name in globals(): del globals()[name]
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()
print(f"Kernel VRAM after cleanup: {torch.cuda.memory_allocated()/1e9:.1f} GB")


In [ ]:
from pathlib import Path
p = Path('src/cxr_intel/eval/metrics_report.py')
content = p.read_text()
new = content.replace(
    '''def _compute_bleu(refs: Sequence[str], preds: Sequence[str]) -> tuple[float, float, float]:
    import sacrebleu

    bleu1 = sacrebleu.corpus_bleu(preds, [refs], max_ngram_order=1).score / 100
    bleu2 = sacrebleu.corpus_bleu(preds, [refs], max_ngram_order=2).score / 100
    bleu4 = sacrebleu.corpus_bleu(preds, [refs]).score / 100
    return bleu1, bleu2, bleu4''',
    '''def _compute_bleu(refs: Sequence[str], preds: Sequence[str]) -> tuple[float, float, float]:
    from sacrebleu.metrics import BLEU

    bleu1 = BLEU(max_ngram_order=1, effective_order=True).corpus_score(preds, [refs]).score / 100
    bleu2 = BLEU(max_ngram_order=2, effective_order=True).corpus_score(preds, [refs]).score / 100
    bleu4 = BLEU(effective_order=True).corpus_score(preds, [refs]).score / 100
    return bleu1, bleu2, bleu4'''
)
if new != content:
    p.write_text(new)
    print("✓ Patched metrics_report.py to use BLEU class API")
else:
    print("⚠ No match — check source")


In [ ]:
from pathlib import Path
s = Path('scripts/run_eval.py')
content = s.read_text()
new = content.replace(
    'medgemma = MedGemmaRunner()\n    medgemma.load()',
    "medgemma = MedGemmaRunner(quantization='int4')\n    medgemma.load()"
)
if new != content:
    s.write_text(new)
    print("✓ Patched run_eval.py: MedGemma uses INT4 quantization")
else:
    print("⚠ No replacement made — let me know to send a manual patch")


In [ ]:
!python -u scripts/run_eval.py --mode report --configs medgemma_only,biomedclip_rag,colpali_zs_rag --test-size 5 --top-k 3


In [11]:
import pandas as pd
df = pd.read_csv('results/tables/report_metrics.csv')
print(df.to_string(index=False))


        config    bleu1    bleu2    bleu4  rouge_l  bertscore_f1  chexbert_f1  radgraph_f1  n
 medgemma_only 0.109534 0.064433 0.019285 0.224373      0.846989     0.133333          NaN  5
biomedclip_rag 0.305137 0.261527 0.215319 0.404095      0.877044     0.300000          NaN  5
colpali_zs_rag 0.241381 0.196534 0.140760 0.413585      0.878289     0.300000          NaN  5


In [14]:
!python -u scripts/run_eval.py \
    --mode report --mode qa --mode retrieval \
    --configs medgemma_only,biomedclip_rag,colpali_zs_rag \
    --test-size 50 --top-k 3


17:04:25 | INFO    | cxr_intel.models.medgemma_runner | Loading MedGemma google/medgemma-4b-it (dtype=bfloat16, quant=int4)
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|█████████████████████| 772/772 [00:00<00:00, 19101.80it/s]
[transformers] DebertaModel LOAD REPORT from: microsoft/deberta-xlarge-mnli
Key                 | Status     |  | 
--------------------+------------+--+-
pooler.dense.weight | UNEXPECTED |  | 
classifier.weight   | UNEXPECTED |  | 
pooler.dense.bias   | UNEXPECTED |  | 
classifier.bias     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
17:13:15 | WARNING | cxr_intel.eval.metrics_report | BERTScore failed (int too big to convert) — falling back to roberta-large
Loading weights: 100%|███████████████████████| 389/389 [00:00<00:00, 976.18it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Sta

In [15]:
from pathlib import Path
s = Path('scripts/run_eval.py')
content = s.read_text()
new = content.replace(
    'judge = LLMJudge(LLMRouter())',
    "judge = LLMJudge(LLMRouter(model='meta/llama-3.3-70b-instruct'))"
)
if new != content:
    s.write_text(new)
    print("✓ Patched judge to use meta/llama-3.3-70b-instruct (NIM naming)")
else:
    print("⚠ No replacement made — search for 'LLMJudge' manually")


✓ Patched judge to use meta/llama-3.3-70b-instruct (NIM naming)


In [18]:
!python -u scripts/run_eval.py \
    --mode qa --mode retrieval \
    --configs medgemma_only,biomedclip_rag,colpali_zs_rag \
    --test-size 50 --top-k 3


18:04:50 | INFO    | cxr_intel.models.medgemma_runner | Loading MedGemma google/medgemma-4b-it (dtype=bfloat16, quant=int4)
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
qa::medgemma_only: 100%|████████████████████████| 50/50 [05:25<00:00,  6.52s/it]
18:10:28 | INFO    | cxr_intel.models.llm_router | LLMRouter using provider=nvidia model=meta/llama-3.1-8b-instruct
18:11:22 | WARNING | cxr_intel.eval.llm_judge | Judge error: RetryError[<Future at 0x7b0d73620e90 state=finished raised RateLimitError>]
Loading weights: 100%|██████████████████████| 389/389 [00:00<00:00, 2844.01it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 


In [19]:
import pandas as pd
for name in ['report_metrics', 'qa_metrics', 'retrieval_metrics']:
    print(f'\n=== {name} ===')
    print(pd.read_csv(f'results/tables/{name}.csv').to_string(index=False))



=== report_metrics ===
        config    bleu1    bleu2    bleu4  rouge_l  bertscore_f1  chexbert_f1  radgraph_f1  n
 medgemma_only 0.000740 0.000490 0.000214 0.185579      0.843089     0.300778          NaN 50
biomedclip_rag 0.002391 0.001829 0.001152 0.261160      0.863751     0.352333          NaN 50
colpali_zs_rag 0.003564 0.002711 0.001642 0.259949      0.865022     0.428788          NaN 50

=== qa_metrics ===
        config  exact_match  token_f1  bertscore_f1  llm_judge_mean  llm_judge_pass_rate  n
 medgemma_only          0.0  0.259093      0.887048            3.42                 0.66 50
biomedclip_rag          0.0  0.448314      0.909041            4.39                 0.88 50
colpali_zs_rag          0.0  0.453927      0.911461            4.34                 0.90 50

=== retrieval_metrics ===
   backend  recall_at_1  recall_at_5  recall_at_10    mrr  ndcg_at_10  n                       by_k
biomedclip          0.0          0.0          0.02 0.0025    0.006309 50 {1: 0.0, 5: 

In [20]:
import shutil
shutil.make_archive('/kaggle/working/results', 'zip', 'results')
print("Download /kaggle/working/results.zip from the Output panel")


Download /kaggle/working/results.zip from the Output panel


In [17]:
from pathlib import Path
s = Path('scripts/run_eval.py')
content = s.read_text()
new = content.replace(
    "judge = LLMJudge(LLMRouter(model='meta/llama-3.3-70b-instruct'))",
    "judge = LLMJudge(LLMRouter(model='meta/llama-3.1-8b-instruct'))"
)
s.write_text(new)
print("✓ Switched judge to llama-3.1-8b (less rate-limited)")


✓ Switched judge to llama-3.1-8b (less rate-limited)
